# Customer Support Chatbot — Student Notebook

## 🎯 Session Goal
Build a **Retrieval-Augmented Generation (RAG)** FAQ chatbot in 7 steps:
1. Load 500-row FAQ dataset from CSV
2. Chunk & embed with OpenAI
3. Store vectors in Chroma (persistent database)
4. Build a RAG chain (retriever → prompt → LLM)
5. Test with real questions

## 📋 Prerequisites
- `.env` file with `OPENAI_API_KEY` set
- Run `python data/generate_faq.py` once to create the FAQ CSV

## ⚡ Live Coding Session
Follow each step below. **Do not skip cells.** Run top-to-bottom once.

---

## Step 1: Imports & Setup

**What we'll do:**
- Import LangChain tools for LLMs, embeddings, text splitting, and vector stores
- Import Pandas for data loading
- Load environment variables (API keys) from `.env`
- Initialize the embedder (OpenAI text-embedding-3-small) and LLM (GPT-4o-mini)

**Key concepts:**
- **Embedder**: Converts text → vectors for semantic search
- **LLM**: The language model answering questions
- **RecursiveCharacterTextSplitter**: Chunks long docs smartly
- **Chroma**: Persistent vector store (survives kernel restarts)

**Your task:** Write code to import all libraries and initialize the LLM + embedder.

In [18]:
# STEP 1: Write your imports and setup here
#import os, load_dotenv, pandas, pathlib

import os
from dotenv import load_dotenv
import pandas as pd
from getpass import getpass

# Load .env if present
load_dotenv()

# Load the Openai+langsmith skeys:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or getpass("OPENAI_API_KEY:")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGSMITH_API_KEY")
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT", "CUSTOMER_SUPPORT_PROJECT")
os.environ['LANGCHAIN_TRACING_V2'] = "true"

print("LangSmith Tracing is ON for this Notebook")


LangSmith Tracing is ON for this Notebook


In [19]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma

from langchain_core.documents import Document #Wrapper class that holds text content + metadata (source, tags, etc.,)

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder #ChatPromptTemplate - Format prompts with variables # MessagesPlaceholder = ALoows dynamic insersation of conversation history

from langchain_core.output_parsers import StrOutputParser # Extract text from LLM output responses

from langchain_core.runnables import RunnablePassthrough # Passes the inputs unchanged

from langchain_core.runnables.history import RunnableWithMessageHistory # Manages converation memory by storing and retriving the chat history per session

from langchain_core.chat_history import InMemoryChatMessageHistory # Stores chat messages in memory


In [20]:

embedder = OpenAIEmbeddings(model = "text-embedding-3-small")

llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)

print("Step 1: Ready to code imports & setup")

Step 1: Ready to code imports & setup


---

## Step 2: Verify API Keys

**What we'll do:**
- Assert that `OPENAI_API_KEY` is set (either in `.env` or as environment variable)
- Print a success message if the key is found

**Why:** No key = no LLM calls. Fail fast!

**Your task:** Write an assertion to check the key exists.

In [21]:
# STEP 2: Check that OPENAI_API_KEY exists
# assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY!"
# print("✅ API key loaded.")

print("Step 2: Ready to verify keys")

Step 2: Ready to verify keys


---

## Step 3: Load the FAQ Dataset

**What we'll do:**
- Load `data/customer_support_faq.csv` using Pandas
- Convert each row (question + answer) into a LangChain `Document` object
- Keep metadata: region, channel, issue (for filtering later)

**Key concept:**
- **Document**: A container with `page_content` (text) and `metadata` (tags)
- Each FAQ row becomes one Document object

**Your task:** Load CSV, convert to Documents with metadata.

In [22]:
# STEP 3: Load the FAQ dataset
# csv_path = Path("data/customer_support_faq.csv")
# faq_df = pd.read_csv(csv_path)
# documents = [
#     Document(
#         page_content=f"Q: {row['question']}\nA: {row['answer']}",
#         metadata={"source": "faq_csv", "row": idx, "region": row.get("region"), ...}
#     )
#     for idx, row in faq_df.iterrows()
# ]

import pandas as pd
from pathlib import Path # Cross-platform file path handling

#csv_path = Path("data/customer_support_faq.csv")

faq_df = pd.read_csv("data/customer_support_faq.csv")

faq_items = faq_df.to_dict(orient = "records") # Convert the Dataframe rows to list of dicts for much easier iterations



In [23]:
# Turn rows into document objects for retrieval
# Document objects allow embedding and storing metadata alongside the text content
# This code is converting your DF into LangChain Document Objects. We are not changing any strucutre here, we are just bringing it in LangChain Docuemnt Objects.

from langchain_core.documents import Document 

documents = [
    Document(
    
    page_content=f"Q:{row['question']}\nA:{row['answer']}",
    
    metadata={"source":"faq_csv", "row": idx, "region": row.get("region"),
              "chennel":row.get("channel"), "issue": row.get("issue")}
)
for idx, row in faq_df.iterrows() # Iterate through each FAQ rows
]

len(documents)

500

---

## Step 4: Build a Persistent Vector Store (Chroma)

**What we'll do:**
- Embed all chunks using the OpenAI embedder
- Store embeddings + text in **Chroma** at `data/chroma_faq/`
- Create a `retriever` to fetch relevant chunks for a query

**Key concepts:**
- **Chroma**: Persistent vector database—embeddings survive kernel restarts!
- **Retriever**: Finds the K most similar chunks to a user query (semantic search)

**Demo after coding:** Test with `retriever.get_relevant_documents("password reset")` to see top matches.

### No Chunking:



In [24]:
# STEP 5: Build Chroma vector store

persist_dir = "data/chroma_faq"

vectorstore = Chroma.from_documents(
    documents = documents, # User Original Complete Q & A Pairs (Not Chunks)
    embedding = embedder,
    persist_directory=persist_dir
)

retriever = vectorstore.as_retriever() # Interface to query and the vector store

print(f"✅ Chroma vector store ready at {persist_dir}")

✅ Chroma vector store ready at data/chroma_faq


# Memory Store
- store = {} is like a memory bank, each customer must have an isolated memory. No cross-talk
2. get_session_history() - Get or create session history: New Customer -> new Memory is created; Existing Customer -> Continue the conversation
## Its like opening a support ticket -> History follows the ticket, not the agent

In [25]:
store = {} # Dictionary to track per-session chat histories : {session_id - > InMemoryChatMessageHistory}

# New Customer -> New Memory
# Existing Customer -> Continue the Conversation
def get_session_history(session_id: str):
    """Get or Create chathistory for a session"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

template = """You are a helpful customer support chatbot.
Use the following context to answer the question.
If you don't know, say you don't know.

Context: {context}  # Retrieved FAQ chunks inserted here

Question: {question}  # User's current question

Answer:""" # LLM generated response based on context + history


prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    MessagesPlaceholder(variable_name = "history"), ## injects past conversations
    ("human", "{question}")
])


# This function transformes retriever output into LLM-ready text.
# Taks a list of document objects and converts them into a single readable string.
# The retriever returns Documents Objects
# LLM understands plain text
# 

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

from langchain_core.runnables import RunnableParallel
# Orchestrator Engine - 
# 1. Runs multiple operations at the same time
# 2. Combines results into one dictionary

# RunnableParallel: Executes all 3 tasks simultaneously, combines results into single dict
# Prepare 3 pieces

base_chain = (
    RunnableParallel( 
    context = lambda x: format_docs(retriever.invoke(x.get("question", ""))),
    question = lambda x: x.get("question", ""),
    history = lambda x: x.get("history", [])
)
| prompt
| llm
| StrOutputParser()
)

# Saves every user message
# Saves every AI response
# Reloads history on next turn

qa_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key= "question",
    history_messages_key = "history"
)



#### User Question -> Load Session History -> Retrieve Relevant FAQs -> Build RAG Prompt -> LLM Generates Answer -> Save Conversation to Memory

In [ ]:
session_id = "user_session_100"

## helper funtion to chat with memory
def chat_with_memory(user_input, session_id = session_id):
    """
    Ask a question with conversation history, 
    1. pass the question to q_chain
    2. Chain retrieves FAQ context + loads history for session_id
    3. LLM answers using context+history
    4. New Q&A automatically saved to session history
    5. All steps traced in LangSmith
    """
    config = {"configurable": {"session_id": session_id}} # Tell RunnableWithMessage History which session to use
    answer = qa_chain.invoke({"question": user_input}, config = config) # Loads past conversations, Retrieves relevent FAQs, Build RAG Prompt, Calls LLMs
    return answer

questions = [
      "How do I handle reset password via the web app?",
      "How do I handle update payment method via the web app?",
      "How do I handle two-factor auth via the mobile app?"
]

for i, q in enumerate(questions, 1):
    answer = chat_with_memory(q)
    print(f"Turn {i}")
    print(f"Customer: {q}")
    print(f"Support Bot: {answer}\n")

Tuen 1
Customer: How do I handle reset password via the web app?
Support Bot: For reset password on the web app, open Settings > Security and follow the prompts. If it persists, open a ticket with logs, region=US, and a short description. Typical resolution: 24h.

Tuen 2
Customer: How do I handle update payment method via the web app?
Support Bot: For update payment method on the web app, visit Billing > Payments and refresh your card on file. If it persists, open a ticket with logs, region=EU, and a short description. Typical resolution: 24h.

Tuen 3
Customer: How do I handle two-factor auth via the mobile app?
Support Bot: For two-factor auth on the mobile app, open Settings > Security and follow the prompts. If it persists, open a ticket with logs, region=US, and a short description. Typical resolution: 24h.



User Question -> Session ID -> Load Conversation History -> Retrieve Relevant FAQs -> Build RAG PRompt -> LLM Answer -> Save History -> Trace Every Thing

---

## Step 6: Build the RAG Chain

**What we'll do:**
- Create a **Prompt Template** with placeholders: `{context}` and `{question}`
- Wire together:
  1. **Retrieval**: Get relevant chunks for the user's question
  2. **Prompt**: Fill in context + question
  3. **LLM**: Generate an answer
  4. **Parse**: Extract the text response

**RAG pipeline flowchart:**
```
User Question
    ↓
[Retriever] → Find top-K relevant chunks
    ↓
[Prompt Template] → Format: "You are a support bot. Context: {chunks}. Q: {q}. Answer:"
    ↓
[LLM (GPT-4o-mini)] → Generate response
    ↓
[StrOutputParser] → Return plain text
```

**Key point:** `RunnableParallel` fetches context and question in parallel before feeding to LLM.

In [ ]:
# STEP 6: Build the RAG chain
# template = """You are a helpful customer support chatbot.
# Use the context to answer the question.
# If you don't know, say you don't know.
#
# Context: {context}
# Question: {question}
# Answer:"""
#
# prompt = ChatPromptTemplate.from_template(template)
#
# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)
#
# rag_chain = (
#     RunnableParallel(
#         context=lambda x: format_docs(retriever.invoke(x.get("question", ""))),
#         question=lambda x: x.get("question", "")
#     )
#     | prompt
#     | llm
#     | StrOutputParser()
# )
#
# print("✅ RAG chain ready!")

print("Step 6: Ready to build RAG chain")

---

## Step 7: Test with Real Questions

**What we'll do:**
- Ask the chain a question like: _"How do I reset my password via the mobile app?"_
- Watch it retrieve relevant FAQ chunks and generate an answer

**What happens under the hood:**
1. User question is embedded
2. Retriever finds top-K matching chunks from Chroma
3. Chunks are formatted into the prompt
4. LLM reads prompt + context and generates a response

**Try these questions:**
- "How do I reset my password via the mobile app?"
- "What is your refund policy?"
- "Do you offer phone support?"
- (Make up your own!)

In [ ]:
# STEP 7: Test the chain with questions
# questions = [
#     "How do I reset my password via the mobile app?",
#     "What is your refund policy?",
#     "Do you offer phone support?"
# ]
#
# for q in questions:
#     answer = rag_chain.invoke({"question": q})
#     print(f"Q: {q}")
#     print(f"A: {answer}\n")

print("Step 7: Ready to test the chatbot")

---

## Advanced: Memory Strategies (Steps 8-10)

### Why Different Memory Strategies?

Long conversations get expensive—every API call includes the full history. Here are three approaches to manage conversation context:

**1. Buffer Memory**: Keep ALL messages (full context, highest token cost)
**2. Window Memory**: Keep only LAST N messages (balanced, fixed cost)
**3. Summary Memory**: Summarize old messages with LLM (lowest cost, may lose detail)

Each has trade-offs. Choose based on your use case!

---

## Step 8: Buffer Memory (All Messages)

**What we'll do:**
- Initialize a separate in-memory store for this strategy
- Keep ALL conversation messages
- Show how memory size grows with each turn
- Perfect for short, focused conversations

**Key concept:**
- **Buffer**: No message loss, but token usage scales linearly
- **Use case**: Customer support chats (typically 3-5 turns)

**Your task:** Implement buffer memory storage and chat loop.

In [ ]:
# STEP 8: Buffer Memory (Keep ALL messages)
# store_buffer = {}
#
# def get_session_history_buffer(session_id: str):
#     if session_id not in store_buffer:
#         store_buffer[session_id] = InMemoryChatMessageHistory()
#     return store_buffer[session_id]
#
# qa_chain_buffer = RunnableWithMessageHistory(
#     base_chain,
#     get_session_history_buffer,
#     input_messages_key="question",
#     history_messages_key="history"
# )
#
# def chat_buffer(user_input, session_id="buffer_session"):
#     config = {"configurable": {"session_id": session_id}}
#     return qa_chain_buffer.invoke({"question": user_input}, config=config)
#
# # Test it
# for i, q in enumerate(questions, 1):
#     answer = chat_buffer(q)
#     history_size = len(store_buffer["buffer_session"].messages)
#     print(f"Turn {i}: {q}")
#     print(f"Bot: {answer}")
#     print(f"Memory: {history_size} messages\n")

print("Step 8: Ready to implement buffer memory")

---

## Step 9: Window Memory (Last N Messages)

**What we'll do:**
- Keep only the LAST N messages (e.g., last 4 messages = 2 conversation turns)
- Discard older context automatically
- Show fixed token cost regardless of conversation length

**Key concepts:**
- **Window size**: How many recent messages to keep (e.g., 4 = last 2 turns)
- **Trade-off**: Loses older context but cost is predictable
- **Use case**: Long chatbots where you only need recent context

**Why 4 messages?** 1 turn = 1 user + 1 assistant = 2 messages. Window of 4 = keep last 2 turns.

**Your task:** Implement window-based history trimming.

In [ ]:
# STEP 9: Window Memory (Keep last N messages only)
# from langchain_core.messages import HumanMessage, AIMessage
#
# store_window = {}
# window_size = 4  # Keep last 4 messages (2 turns)
#
# def get_session_history_window(session_id: str):
#     if session_id not in store_window:
#         store_window[session_id] = InMemoryChatMessageHistory()
#     return store_window[session_id]
#
# def trim_window_history(history_obj, window_size=4):
#     """Keep only last N messages"""
#     messages = history_obj.messages
#     if len(messages) > window_size:
#         return messages[-window_size:]
#     return messages
#
# # Custom chain wrapper for window memory
# def chat_window(user_input, session_id="window_session"):
#     history_obj = get_session_history_window(session_id)
#     config = {"configurable": {"session_id": session_id}}
#     answer = qa_chain.invoke({"question": user_input}, config=config)
#     
#     # Trim history to window size
#     trimmed = trim_window_history(history_obj, window_size)
#     history_obj.messages = trimmed
#     return answer
#
# # Test it
# for i, q in enumerate(questions, 1):
#     answer = chat_window(q)
#     history_size = len(store_window["window_session"].messages)
#     print(f"Turn {i}: {q}")
#     print(f"Bot: {answer}")
#     print(f"Memory: {history_size}/{window_size} messages\n")

print("Step 9: Ready to implement window memory")

---

## Step 10: Summary Memory (LLM-Based Compression)

**What we'll do:**
- Use the LLM itself to SUMMARIZE old messages
- Keep recent N turns unmodified (full detail)
- Replace older turns with 1-2 sentence summaries
- Most efficient for very long conversations

**Key concepts:**
- **keep_recent_messages**: How many recent turns to preserve fully (e.g., 2)
- **Summarization**: Use LLM to compress old turns into key points
- **Use case**: Research sessions, long document Q&A, multi-hour support chats

**Trade-off:** Cheapest but loses granular detail from old context.

**Your task:** Implement LLM-based message compression.

In [ ]:
# STEP 10: Summary Memory (LLM-based compression)
# store_summary = {}
# keep_recent_messages = 2  # Always keep last 2 turns, summarize the rest
#
# def get_session_history_summary(session_id: str):
#     if session_id not in store_summary:
#         store_summary[session_id] = {
#             "messages": InMemoryChatMessageHistory(),
#             "summary": ""
#         }
#     return store_summary[session_id]["messages"]
#
# def compress_with_llm(messages, llm, keep_recent=2):
#     """Use LLM to summarize old messages"""
#     if len(messages) <= keep_recent * 2:
#         return None  # Not enough messages to summarize
#     
#     old_messages = messages[:-keep_recent*2]
#     old_text = "\n".join([msg.content for msg in old_messages if hasattr(msg, 'content')])
#     
#     summary_prompt = f"""Summarize this customer support conversation in 1-2 sentences, focusing on key issues:
# {old_text}
# Summary:"""
#     
#     summary = llm.invoke(summary_prompt)
#     return summary.content if hasattr(summary, 'content') else str(summary)
#
# qa_chain_summary = RunnableWithMessageHistory(
#     base_chain,
#     get_session_history_summary,
#     input_messages_key="question",
#     history_messages_key="history"
# )
#
# def chat_summary(user_input, session_id="summary_session"):
#     if session_id not in store_summary:
#         get_session_history_summary(session_id)
#     
#     config = {"configurable": {"session_id": session_id}}
#     answer = qa_chain_summary.invoke({"question": user_input}, config=config)
#     
#     history_obj = store_summary[session_id]["messages"]
#     if len(history_obj.messages) > keep_recent_messages * 2:
#         summary = compress_with_llm(history_obj.messages, llm, keep_recent=keep_recent_messages)
#         if summary:
#             store_summary[session_id]["summary"] = summary
#     
#     return answer
#
# # Test it
# for i, q in enumerate(questions, 1):
#     answer = chat_summary(q)
#     history = store_summary["summary_session"]["messages"].messages
#     summary = store_summary["summary_session"]["summary"]
#     print(f"Turn {i}: {q}")
#     print(f"Bot: {answer}")
#     if summary:
#         print(f"Compressed: {summary[:80]}...")
#     print()

print("Step 10: Ready to implement summary memory")

---

## Memory Strategy Comparison

| Strategy | Keep | Token Cost | Context Loss | Best For |
|----------|------|-----------|--------------|----------|
| **Buffer** | All messages | High ↑ | None | Short chats (3-5 turns), need full history |
| **Window** | Last N msgs | Fixed ↓ | Older turns lost | Long chatbots, balance cost & context |
| **Summary** | Recent + compressed | Low ↓↓ | Granular detail lost | Very long chats, efficiency critical |

### When to Choose Each:
- **Buffer**: Classic customer support (short, focused)
- **Window**: Multi-turn chatbots (medium depth, 10-20 turns)
- **Summary**: Research sessions, document Q&A (unlimited turns)

---

## 🚀 Bonus Exercises

**If you finish early:**

1. **Compare memory sizes:** Print out the message count for each strategy at the end. Which uses fewer tokens?

2. **Test with longer conversation:** Add 5-10 more questions. See how buffer memory grows vs window/summary.

3. **Tweak window size:** Try `window_size=2` (only last turn) vs `window_size=8` (last 4 turns). How does context quality change?

4. **Custom summary style:** Modify the summary prompt to focus on specific topics (e.g., billing issues only).

5. **Measure latency:** Time how long each memory strategy takes. Is summary slower due to LLM compression?

6. **Enable LangSmith:** Trace all three strategies in the LangSmith dashboard. Compare token usage!

---

## ✅ Summary

**You've now mastered RAG + Memory in 10 steps!**
- ✅ Built a full RAG pipeline (CSV → chunks → embeddings → retriever → LLM)
- ✅ Implemented 3 memory strategies (Buffer, Window, Summary)
- ✅ Learned cost vs context trade-offs
- ✅ Ready for production customer support chats

**Next level:** Deploy with FastAPI, add multi-user sessions, integrate LangSmith monitoring.